# Leak handedness on a cohort

## Question and why it matters
The compact ant keeps its territory compact when the turn on fresh ground is bounded (forced R, forced L, reversal) and goes
porous when that turn is random R/L (`experiments/active/2026-10-wall-vs-memory/`). The leak titration
(`experiments/active/2026-10-leak-titration/`) raised the random share EF gradually and, on three rules, found that a forced-L
base seemed to need more leak than forced R before compactness was lost. If that holds, the bound on fresh runs is not the whole
story: the two arms have the same fresh-run law at every EF. This notebook scores that lead on 77 + 9 rules (Q42).
Registration, gates and predictions: `README.md`; lab record: `NOTEBOOK.md`.

## Assumptions and definitions
* Leak EF: on a never-departed cell the forced base turn is replaced by R or L with probability 1/2 each, so the minority turn
  has probability EF/2 (20-slot `fs:` set). Arm R = base R (the ant's own colour-0 turn), arm L = base L.
* Early a: slope of log area on log t over windows k = 12..22 (probe, board 16384); classes as in wall-vs-memory.
* Rise fraction F(EF) = (a(EF) - a(0)) / (a(1) - a(0)) per arm, with a shared EF = 1 level (same law in both arms; gate G2).
* Lag Lambda = mean over EF 0.3, 0.5 of F_R - F_L. Positive: arm L has risen less (tolerates the leak better).

## Provenance
Git commit, date and SHA-256 of every input. Seeds 2026111201..04 (in the specs).

In [1]:
import hashlib, subprocess, datetime
from pathlib import Path
import numpy as np
import polars as pl

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
EXP = ROOT / "experiments/active/2026-11-leak-handedness-cohort"
INPUTS = [EXP / f"results/{n}" for n in ("specs.txt", "cohorts.csv", "probe.csv", "runs.csv", "rules.csv",
                                         "analysis.txt", "calibration.txt", "posthoc.txt", "provenance.txt")]
sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
print("git", sha, "| run", datetime.date.today().isoformat())
for p in INPUTS:
    print(hashlib.sha256(p.read_bytes()).hexdigest()[:16], p.relative_to(ROOT))

git 3976f1b | run 2026-11-12
41b1a9b4850aae41 experiments/active/2026-11-leak-handedness-cohort/results/specs.txt
e760b37f45c5c856 experiments/active/2026-11-leak-handedness-cohort/results/cohorts.csv
e09b8ea6fce30d60 experiments/active/2026-11-leak-handedness-cohort/results/probe.csv
35ece613da3f7a27 experiments/active/2026-11-leak-handedness-cohort/results/runs.csv
04028922f19a38ec experiments/active/2026-11-leak-handedness-cohort/results/rules.csv
376005a58e81b701 experiments/active/2026-11-leak-handedness-cohort/results/analysis.txt
9904e364aa7524ef experiments/active/2026-11-leak-handedness-cohort/results/calibration.txt
835999cd0398660a experiments/active/2026-11-leak-handedness-cohort/results/posthoc.txt
7eba2e09ac6bea11 experiments/active/2026-11-leak-handedness-cohort/results/provenance.txt


## Gates and registered scores
The frozen scorer's output, unedited.

In [2]:
print((EXP / "results/analysis.txt").read_text())

# leak-handedness cohort scoring; 2708 specs
G1 replication: 172/172 EF = 0 rows identical to Stage 3b -> PASS
classes by cohort/arm/EF (EF > 0):
  C L EF 0.1: {'compact': 297, 'intermediate': 11}
  C L EF 0.3: {'compact': 170, 'intermediate': 138}
  C L EF 0.5: {'compact': 42, 'intermediate': 218, 'porous': 48}
  C L EF 1.0: {'escape': 26, 'intermediate': 69, 'porous': 213}
  C R EF 0.1: {'compact': 301, 'intermediate': 7}
  C R EF 0.3: {'compact': 174, 'intermediate': 125, 'porous': 9}
  C R EF 0.5: {'compact': 43, 'escape': 2, 'intermediate': 215, 'porous': 48}
  C R EF 1.0: {'escape': 36, 'intermediate': 53, 'porous': 219}
  E L EF 0.1: {'compact': 14, 'escape': 3, 'intermediate': 10, 'porous': 9}
  E R EF 0.1: {'compact': 35, 'intermediate': 1}
cohort C: 77 rules, scoreable 77
  not scoreable: 
G2 shared law at EF = 1: mean d -0.0015 (need |.| <= 0.02); |d| <= 0.06 in 1.000 of rules (need >= 0.90) -> PASS
P1 handedness lag: median Lambda +0.0234, rule-bootstrap 95% [-0.1582, +0.18

Both gates pass: the EF = 0 rows replicate Stage 3b byte for byte, and the two arms agree at EF = 1 (same law).
**P1 is unresolved** and **P2 failed** (3 of the 9 forced-L escaping rules collapse at a 10% leak, not most of them).

## Why P1 is unresolved: the per-rule lags are large and of both signs

In [3]:
r = pl.read_csv(EXP / "results/rules.csv").sort("lam")
lam = r["lam"].to_numpy()
print(f"Lambda across 77 rules: sd {lam.std(ddof=1):.3f}, quartiles {np.quantile(lam, .25):+.3f} / {np.quantile(lam, .75):+.3f}")
hist, edges = np.histogram(lam, bins=np.arange(-0.9, 1.0, 0.1))
for h, e in zip(hist, edges):
    print(f"{e:+.1f} .. {e + 0.1:+.1f} | " + "#" * h)
print((EXP / "results/calibration.txt").read_text())

Lambda across 77 rules: sd 0.383, quartiles -0.290 / +0.303
-0.9 .. -0.8 | 
-0.8 .. -0.7 | #
-0.7 .. -0.6 | ##
-0.6 .. -0.5 | ###
-0.5 .. -0.4 | ###
-0.4 .. -0.3 | #########
-0.3 .. -0.2 | ########
-0.2 .. -0.1 | #######
-0.1 .. -0.0 | ###
-0.0 .. +0.1 | #####
+0.1 .. +0.2 | #######
+0.2 .. +0.3 | #########
+0.3 .. +0.4 | #####
+0.4 .. +0.5 | #####
+0.5 .. +0.6 | ####
+0.6 .. +0.7 | ####
+0.7 .. +0.8 | 
+0.8 .. +0.9 | ##
# P1 decision-rule calibration, synthetic, seed 20261112, 70 rules, 1000 draws, 2000 bootstrap resamples
# both arms share one EF = 1 level per rule (same law); EF = 0 levels differ by arm
sd 0.025 true lag +0.00: no lag 0.953, reversed 0.024, unresolved 0.023
sd 0.025 true lag +0.05: lag 0.516, no lag 0.029, unresolved 0.455
sd 0.025 true lag +0.10: lag 1.000
sd 0.025 true lag +0.20: lag 1.000
sd 0.025 true lag -0.10: reversed 1.000
sd 0.040 true lag +0.00: lag 0.003, no lag 0.699, reversed 0.026, unresolved 0.272
sd 0.040 true lag +0.05: lag 0.466, no lag 0.025, unre

Seed noise on one rule's Lambda is about 0.05 (see the post hoc file below), so a spread of 0.38 is real rule-to-rule
variation. The calibration assumed one shared lag (spread 0.05) and therefore expected a narrow interval; that assumption
was wrong, and a cohort median cannot see a lag whose sign depends on the rule. This does not show that handedness is
irrelevant; it shows the opposite for most rules, with the sign varying.

## Post hoc (not registered): the sign tracks rule composition

In [4]:
print((EXP / "results/posthoc.txt").read_text())
r2 = r.with_columns(fracR=pl.col("rule").map_elements(lambda s: s.count("R") / len(s), return_dtype=pl.Float64))
with pl.Config(tbl_rows=12):
    print(r2.select("rule", "fracR", "lam").head(6))
    print(r2.select("rule", "fracR", "lam").tail(6))

# POST HOC, not registered. Cohort C, 77 rules.
Lambda: mean +0.029, sd 0.383, quartiles -0.290 / +0.303, min -0.781, max +0.840; |Lambda| > 0.4 in 24 rules
median within-rule per-seed sd of early a at EF 0.3/0.5: 0.0196; median Delta_R 0.196; implied seed noise on one rule's Lambda ~ 0.050 (sd/sqrt(4)/Delta, x sqrt(2) arms, / sqrt(2) EFs)
Spearman(Lambda, fraction of R letters) = +0.404, p = 0.00027 (one of several looks; uncorrected)
  R fraction in [0.00, 0.45): n 26, median Lambda -0.205, Lambda > 0 in 9
  R fraction in [0.45, 0.55): n 17, median Lambda +0.123, Lambda > 0 in 10
  R fraction in [0.55, 1.00): n 34, median Lambda +0.175, Lambda > 0 in 22
  prefix RLLL: n 14, median Lambda -0.384, Lambda > 0 in 2
  prefix RRRR: n 7, median Lambda +0.544, Lambda > 0 in 7

shape: (6, 3)
┌──────────────┬──────────┬───────────┐
│ rule         ┆ fracR    ┆ lam       │
│ ---          ┆ ---      ┆ ---       │
│ str          ┆ f64      ┆ f64       │
╞══════════════╪══════════╪═══════════╡
│ RL

Read literally: the leak does less damage when the forced fresh-ground turn is the rule's minority turn. This was found by
looking at the data after the registered test failed to resolve, composition and prefix are confounded, and it rests on one
cohort. It is a hypothesis for a new registration with a per-rule sign prediction on new rules, not a result.

## Caveats
* Finite window (k = 12..22) on a 16384 board, 4 seeds per cell, two EF points per arm (no EF_50 curve).
* Cohort: Stage 3b's core-class sample; 14 rules excluded because a base was intermediate.
* Escapes at EF = 1 (62 of 616 runs) have no early fit; every rule kept at least 3 of 4 fits where required.

## Next steps
Register a test of "minority forced turn tolerates more leak" with the sign fixed per rule from its table, on rules outside this
cohort, scored by per-rule sign agreement. Q41 (fresh-run lengths in the ant) stays relevant: the run-length law is the same for
both arms, so any lag points beyond the bound.